In [1]:
import os
import glob
import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

RANDOM_STATE = 42

# Concatenação e split temporal (70/30)

Este notebook:
1) Carrega os CSVs por classe (um arquivo `*_features_processed_timestamp.csv` por pasta).
2) Normaliza nomes de colunas (ex.: `Label` → `label`).
3) Faz limpeza simples de valores faltantes (numéricos → 0; textuais → "-").
4) Aplica **subamostragem apenas nas classes dominantes** (configurável).
5) Realiza **split temporal 70/30 por classe** usando a coluna `ts`.
6) Remove `ts` e exporta `*_train.csv` e `*_test.csv`.

In [2]:
# O caminho base é o diretório atual
base_path = '.'

# Lista de pastas de ataque
attack_folders = [
    'analysis',
    'dos',
    'exploits',
    'fuzzers',
    'reconnaissance'
]

# Lista de todos os diretórios a serem processados
data_paths = {
    'Normal': os.path.join(base_path, 'normal')
}

# Adiciona os caminhos de ataque ao dicionário (mesma estrutura do original: ./ataque/<categoria>)
for attack in attack_folders:
    data_paths[attack.capitalize()] = os.path.join(base_path, 'ataque', attack)

print("Caminhos de dados definidos:")
print(data_paths)

# Lista para armazenar todos os DataFrames
all_dataframes = []
loaded_files = []

# Loop para encontrar e ler todos os arquivos '*_features_processed_timestamp.csv'
pattern = '*_features_processed_timestamp.csv'
print(f"\nIniciando leitura dos arquivos '{pattern}'...")

for category_name, folder_path in data_paths.items():
    if not os.path.isdir(folder_path):
        print(f"Aviso: Diretório não encontrado, pulando: {folder_path}")
        continue

    # Usa glob para encontrar o arquivo dentro da pasta
    search_pattern = os.path.join(folder_path, pattern)
    found_files = glob.glob(search_pattern)

    if not found_files:
        print(f"Aviso: Nenhum arquivo '{pattern}' encontrado em: {folder_path}")
        continue

    # Assume que há apenas um arquivo por pasta (como no original)
    file_path = found_files[0]

    try:
        print(f"Lendo: {file_path} ...")
        df_temp = pd.read_csv(file_path, low_memory=False)
        all_dataframes.append(df_temp)
        loaded_files.append(file_path)
    except Exception as e:
        print(f"Erro ao ler {file_path}: {e}")

print("\nLeitura de todos os arquivos concluída.")
print(f"Total de arquivos carregados: {len(loaded_files)}")

Caminhos de dados definidos:
{'Normal': '.\\normal', 'Analysis': '.\\ataque\\analysis', 'Dos': '.\\ataque\\dos', 'Exploits': '.\\ataque\\exploits', 'Fuzzers': '.\\ataque\\fuzzers', 'Reconnaissance': '.\\ataque\\reconnaissance'}

Iniciando leitura dos arquivos '*_features_processed_timestamp.csv'...
Lendo: .\normal\normal_features_processed_timestamp.csv ...
Lendo: .\ataque\analysis\analysis_features_processed_timestamp.csv ...
Aviso: Nenhum arquivo '*_features_processed_timestamp.csv' encontrado em: .\ataque\dos
Aviso: Nenhum arquivo '*_features_processed_timestamp.csv' encontrado em: .\ataque\exploits
Lendo: .\ataque\fuzzers\fuzzers_features_processed_timestamp.csv ...
Lendo: .\ataque\reconnaissance\reconnaissance_features_processed_timestamp.csv ...

Leitura de todos os arquivos concluída.
Total de arquivos carregados: 4


In [3]:
# =========================
# 2) CONCATENAÇÃO + LIMPEZA
# =========================

if not all_dataframes:
    raise RuntimeError("Nenhum DataFrame foi carregado. Verifique os caminhos/pastas.")

final_dataset = pd.concat(all_dataframes, ignore_index=True)

# Normalizar nomes de colunas esperadas
rename_map = {
    "Label": "label",
    "duration": "dur",          # caso algum CSV tenha 'duration' em vez de 'dur'
    "conn_state": "state",      # caso exista essa variação
}
final_dataset = final_dataset.rename(columns={k:v for k,v in rename_map.items() if k in final_dataset.columns})

# Checagens mínimas
required_cols = ["attack_cat", "label"]
for c in required_cols:
    if c not in final_dataset.columns:
        raise KeyError(f"Coluna obrigatória ausente no dataset consolidado: '{c}'")

if "ts" not in final_dataset.columns:
    raise KeyError("Coluna 'ts' não encontrada. Ela é necessária para o split temporal.")

# Garantir tipos
final_dataset["label"] = pd.to_numeric(final_dataset["label"], errors="coerce").fillna(0).astype(int)
final_dataset["attack_cat"] = final_dataset["attack_cat"].astype(str)

# Limpeza: preenche NaN com base no tipo inferido
# - numéricos -> 0
# - textuais  -> "-"
num_cols = final_dataset.select_dtypes(include=[np.number]).columns.tolist()
obj_cols = [c for c in final_dataset.columns if c not in num_cols]

final_dataset[num_cols] = final_dataset[num_cols].replace([np.inf, -np.inf], np.nan).fillna(0)
final_dataset[obj_cols] = final_dataset[obj_cols].fillna("-")

# Garantir ts numérico (float) para ordenação
final_dataset["ts"] = pd.to_numeric(final_dataset["ts"], errors="coerce")
final_dataset = final_dataset.dropna(subset=["ts"]).copy()
final_dataset["ts"] = final_dataset["ts"].astype(float)

print("Dataset consolidado:")
print(" - linhas:", len(final_dataset))
print(" - colunas:", len(final_dataset.columns))
print("\nDistribuição (attack_cat):")
print(final_dataset["attack_cat"].value_counts())

Dataset consolidado:
 - linhas: 34403
 - colunas: 46

Distribuição (attack_cat):
attack_cat
Normal            17279
Reconnaissance     9050
Fuzzers            5256
Analysis           2818
Name: count, dtype: int64


In [4]:
# =========================
# 3) SUBAMOSTRAGEM
# =========================


SUBSAMPLE_FRAC = {
    # Mantém só 1% de Exploits
    "Exploits": 1,
    "DoS": 1,
    "Fuzzers": 1,
    "Reconnaissance": 1, 
    "Analysis": 1 
}

MIN_KEEP_PER_CLASS = 200  # garante um piso (ajuste conforme seu volume)

def subsample_by_fraction(df, cat, frac, min_keep=200, random_state=42):
    df_cat = df[df["attack_cat"] == cat]
    df_other = df[df["attack_cat"] != cat]
    n = len(df_cat)
    if n == 0:
        return df

    target = int(np.floor(n * float(frac)))
    target = max(target, min_keep)
    target = min(target, n)

    if target == n:
        return df

    # Amostra aleatória (reprodutível)
    df_cat_s = df_cat.sample(n=target, random_state=random_state)
    out = pd.concat([df_other, df_cat_s], ignore_index=True)
    return out

before_counts = final_dataset["attack_cat"].value_counts()

for cat, frac in SUBSAMPLE_FRAC.items():
    if cat in final_dataset["attack_cat"].unique():
        n0 = int(before_counts.get(cat, 0))
        final_dataset = subsample_by_fraction(final_dataset, cat, frac, min_keep=MIN_KEEP_PER_CLASS, random_state=RANDOM_STATE)
        n1 = int(final_dataset["attack_cat"].value_counts().get(cat, 0))
        print(f"Subamostragem '{cat}': {n0} -> {n1} (frac={frac})")

print("\nDistribuição após subamostragem (attack_cat):")
print(final_dataset["attack_cat"].value_counts())

Subamostragem 'Fuzzers': 5256 -> 5256 (frac=1)
Subamostragem 'Reconnaissance': 9050 -> 9050 (frac=1)
Subamostragem 'Analysis': 2818 -> 2818 (frac=1)

Distribuição após subamostragem (attack_cat):
attack_cat
Normal            17279
Reconnaissance     9050
Fuzzers            5256
Analysis           2818
Name: count, dtype: int64


In [5]:
# =========================
# 4) SPLIT TEMPORAL 70/30 (por classe)
# =========================
# Para evitar que uma classe inteira caia só no treino ou só no teste
# (com poucas sessões por classe), fazemos o split temporal DENTRO de cada attack_cat.

TRAIN_RATIO = 0.70

train_parts = []
test_parts = []

for cat, df_cat in final_dataset.groupby("attack_cat", sort=False):
    df_cat = df_cat.sort_values("ts").reset_index(drop=True)
    n = len(df_cat)
    if n < 2:
        # muito pequeno: manda tudo pro treino
        train_parts.append(df_cat)
        continue

    cut = int(np.floor(n * TRAIN_RATIO))
    # garante pelo menos 1 no teste quando possível
    cut = min(max(cut, 1), n - 1)

    train_parts.append(df_cat.iloc[:cut].copy())
    test_parts.append(df_cat.iloc[cut:].copy())

df_train = pd.concat(train_parts, ignore_index=True).sort_values("ts").reset_index(drop=True)
df_test  = pd.concat(test_parts, ignore_index=True).sort_values("ts").reset_index(drop=True)

print("Split concluído:")
print(" - treino:", len(df_train))
print(" - teste :", len(df_test))
print("\nDistribuição treino (attack_cat):")
print(df_train["attack_cat"].value_counts())
print("\nDistribuição teste (attack_cat):")
print(df_test["attack_cat"].value_counts())

# Drop ts antes de exportar (como solicitado)
df_train_out = df_train.drop(columns=["ts"], errors="ignore")
df_test_out  = df_test.drop(columns=["ts"], errors="ignore")

output_train = "ubuntu_server_dataset_train.csv"
output_test  = "ubuntu_server_dataset_test.csv"

df_train_out.to_csv(output_train, index=False)
df_test_out.to_csv(output_test, index=False)

print(f"\nCSV treino salvo em: {output_train}")
print(f"CSV teste  salvo em: {output_test}")

display(df_train_out.head())

Split concluído:
 - treino: 24081
 - teste : 10322

Distribuição treino (attack_cat):
attack_cat
Normal            12095
Reconnaissance     6335
Fuzzers            3679
Analysis           1972
Name: count, dtype: int64

Distribuição teste (attack_cat):
attack_cat
Normal            5184
Reconnaissance    2715
Fuzzers           1577
Analysis           846
Name: count, dtype: int64

CSV treino salvo em: ubuntu_server_dataset_train.csv
CSV teste  salvo em: ubuntu_server_dataset_test.csv


,id,dur,proto,service,state,spkts,dpkts,sbytes,dbytes,rate,sttl,dttl,sload,dload,sloss,dloss,sinpkt,dinpkt,sjit,djit,swin,stcpb,dtcpb,dwin,tcprtt,synack,ackdat,smean,dmean,trans_depth,response_body_len,ct_srv_src,ct_state_ttl,ct_dst_ltm,ct_src_dport_ltm,ct_dst_sport_ltm,ct_dst_src_ltm,is_ftp_login,ct_ftp_cmd,ct_flw_http_mthd,ct_src_ltm,ct_srv_dst,is_sm_ips_ports,attack_cat,label
0,0,4.213739,tcp,-,OTH,91.0,0.0,3384.0,0.0,21.596022,0.0,0.0,6424.697875,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000e+00,0.0,0.0,0.0,0.0,37,0,0.0,0.0,1,0,1,1,1,1,0,0,0,1,1,0,Normal,0
1,0,0.401441,tcp,-,CON,1.0,2.0,1514.0,156.0,7.473078,64.0,63.0,30171.307863,3108.800546,0.0,0.0,0.0,0.0,0.0,0.0,510.0,771652313.0,1.725191e+09,30556.0,0.0,0.0,0.0,1514,78,0.0,0.0,1,0,1,1,1,1,0,0,0,1,1,0,Normal,0
2,0,72.549373,tcp,ftp,SH,5.0,0.0,6.0,0.0,0.068919,0.0,0.0,0.661618,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000e+00,0.0,0.0,0.0,0.0,1,0,0.0,0.0,1,0,2,1,1,1,0,0,0,1,1,0,Normal,0
3,0,14.806178,unknown_transport,-,OTH,8.0,0.0,0.0,0.0,0.540315,0.0,0.0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000e+00,0.0,0.0,0.0,0.0,0,0,0.0,0.0,1,0,1,1,1,1,0,0,0,1,1,0,Normal,0
4,0,0.313275,tcp,-,S0,5.0,0.0,43.0,0.0,15.960418,0.0,0.0,1098.076770,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000e+00,0.0,0.0,0.0,0.0,8,0,0.0,0.0,1,0,3,1,1,1,0,0,0,1,2,0,Normal,0
